<a href="https://colab.research.google.com/github/Liran-Martfel/Project-Vision-CNN_-in-progress/blob/Code-updates/PROJECT_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "opencv-python-headless<5"

In [3]:
import zipfile #importing zip actions
from google.colab import drive #library to connect to drive
drive.mount('/content/drive') #connecting with file-path on drive
zipfile_path = '/content/drive/MyDrive/Colab Notebooks/Project - CNN/new_data.zip' #saving the file path under zipfile_path
with zipfile.ZipFile(zipfile_path,'r') as face_exp_data: #from zipfile library, use ZipFile class
  face_exp_data.extractall('/content/data') #extract all data from zipfile
train_data = '/content/data/fer2013plus/fer2013/train'
#since I'm working with drive, i can't use [....] normally, i need to use the path of the files.
test_data = '/content/data/fer2013plus/fer2013/test'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
!rm -rf /content/data/fer2013plus/fer2013/train/contempt /content/data/fer2013plus/fer2013/test/contempt
print(sorted(os.listdir('/content/data/fer2013plus/fer2013/train')))

['anger', 'disgust', 'fear', 'happiness', 'neutral', 'sadness', 'surprise']


In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator #importing the library and action to generate images for the cnn so that he can learn the images

# Training data generator with augmentation
train_datagen = ImageDataGenerator (
    rescale = 1./255, # Normalize pixel values to [0,1] — helps speed up training and stabilize gradients
    shear_range = 0.2,  # Apply a slight diagonal transformation (shear) — simulates natural changes in camera angle
    zoom_range = 0.2,  # Apply random zoom-in effect — helps the model recognize objects at different scales
    horizontal_flip = True # Flip images horizontally — helps the model handle symmetry
)
# Testing data generator — only normalization, no augmentation
test_datagen = ImageDataGenerator(rescale=1./255)

In [6]:
training_set = train_datagen.flow_from_directory( #creating the training set which the cnn will train on with that data
    train_data, #the train data from the exctract ziped file
    target_size = (48,48), # the size of the image
    batch_size = 32, # defualt, the number of images that the model sees all at once.
    class_mode = 'categorical',  #useing categorical beacuse every label is becoming a vector, which will help me when i have more then only 2 chcoices, (like emotions)
    color_mode = 'grayscale')

testing_set = test_datagen.flow_from_directory( #creating the testing set which the cnn will test on with that data
    test_data,
    target_size = (48,48),
    batch_size = 32,
    class_mode = 'categorical',
    color_mode = 'grayscale')

Found 28221 images belonging to 7 classes.
Found 7042 images belonging to 7 classes.


In [ ]:
from keras.models import Sequential #importing the empty model (CNN)
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense,Dropout #importing the 5 types layers of the CNN
cnn = Sequential() #creating the empty CNN model

#first layer, 32 filters of 3X3 weights, 48,48,1 is the original size of the photos & 1 for grayscale
cnn.add(Conv2D (32,(3,3),activation = 'relu', input_shape = (48,48,1))) #define the structure of the layer and add it to the CNN model, using relu as activition as defualt & negatives become 0, also, it's "cheap" and train fast
cnn.add(MaxPooling2D(pool_size = (2,2))) #creating a max_pool with small sqers 2 by 2, and saving the hight number in each sqer

#second layer, 64 filters of 3X3 weights, lose resolution but gain more pattern types.
cnn.add(Conv2D (64,(3,3),activation = 'relu'))
cnn.add(MaxPooling2D(pool_size = (2,2)))

#third layer, 128 filters of 3X3 weights, lose resolution but gain more pattern types.
cnn.add(Conv2D (128,(3,3),activation = 'relu'))
cnn.add(MaxPooling2D(pool_size = (2,2)))
cnn.add(Flatten()) # Flatten, turn the feature maps into one long vector for the layers

#creating the dense
cnn.add(Dense(units = 128,activation= 'relu')) #choose 128 neurons to activate under relu function

#creating a dorpout in order to switch neuron off from the output of the layer before him.
cnn.add(Dropout(0.5)) #0.5 is standart

cnn.add(Dense(units = 7,activation= 'softmax')) #choose 7 neuron beacuse i have 7 emotions to show



In [8]:
cnn.summary() #testing and seeing my CNN model is correct by the way i created him

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 46, 46, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 23, 23, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 21, 21, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 355,847 (1.36 MB)

 Trainable params: 355,847 (1.36 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
from keras.callbacks import EarlyStopping
cnn.compile(optimizer = 'adam', loss = 'categorical_crossentropy',metrics = ['accuracy'])
# stop training when val_loss stops improving for 5 epochs, and keep the best weights
stopper = EarlyStopping(monitor = 'val_loss',patience = 5, restore_best_weights = True)
cnn.fit (x = training_set, validation_data = testing_set, epochs = 50, callbacks = [stopper])


Epoch 1/50
882/882 ━━━━━━━━━━━━━━━━━━━━ 173s 190ms/step - accuracy: 0.3889 - loss: 1.5418 - val_accuracy: 0.5126 - val_loss: 1.3845
Epoch 2/50
882/882 ━━━━━━━━━━━━━━━━━━━━ 165s 187ms/step - accuracy: 0.5319 - loss: 1.2986 - val_accuracy: 0.6142 - val_loss: 1.0837
Epoch 3/50
882/882 ━━━━━━━━━━━━━━━━━━━━ 165s 187ms/step - accuracy: 0.5961 - loss: 1.1373 - val_accuracy: 0.6473 - val_loss: 0.9878
Epoch 4/50
882/882 ━━━━━━━━━━━━━━━━━━━━ 212s 198ms/step - accuracy: 0.6262 - loss: 1.0499 - val_accuracy: 0.6652 - val_loss: 0.9178
Epoch 5/50
882/882 ━━━━━━━━━━━━━━━━━━━━ 162s 183ms/step - accuracy: 0.6485 - loss: 0.9909 - val_accuracy: 0.6785 - val_loss: 0.8825
Epoch 6/50
882/882 ━━━━━━━━━━━━━━━━━━━━ 212s 195ms/step - accuracy: 0.6593 - loss: 0.9517 - val_accuracy: 0.6978 - val_loss: 0.8402
Epoch 7/50
882/882 ━━━━━━━━━━━━━━━━━━━━ 175s 199ms/step - accuracy: 0.6713 - loss: 0.9084 - val_accuracy: 0.7099 - val_loss: 0.8096
Epoch 8/50
882/882 ━━━━━━━━━━━━━━━━━━━━ 162s 183ms/step - accuracy: 0.6821 -

In [19]:
cnn.save('/content/drive/MyDrive/Colab Notebooks/Project - CNN/training_model_v2.keras')
cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 46, 46, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 23, 23, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 21, 21, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,067,543 (4.07 MB)

 Trainable params: 355,847 (1.36 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 711,696 (2.71 MB)

In [24]:
#test rows for the next block of code:
###################################################################################################
#print(img) #<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=447x447 at 0x79C75864E900>
#print(img.size, img.mode)
#print(img.shape, img.dtype, img.min(), img.max())
#print(dir(training_set))
#print (training_set.class_indices) so i can see the emotions order
#prediction_class_name = training_set.class_indices.keys() ,checking for the keys
#print(croped_img.size)
#print(face_detector.empty())

####################################################################################################

In [26]:
from tensorflow.keras.models import load_model
from PIL import Image #Python Image Library
import numpy as np
from google.colab import drive #library to connect to drive
import cv2 # openCV library which have face recegnition trained modelinside

drive.mount('/content/drive') #connecting with file-path on drive
img_test = '/content/drive/MyDrive/Colab Notebooks/Project - CNN/vision-photo.jpg'
cnn_model = load_model('/content/drive/MyDrive/Colab Notebooks/Project - CNN/training_model.keras')
prediction_class_name = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise'] #creating the list of emotions so it can work on the website without the training data

def emotions_probability(prediction_class_name,prediction):
  probability = ""
  for i in range(len(prediction_class_name)):
      probability += f"{prediction_class_name[i]}: {prediction[0][i]* 100:.2f}%\n"
  return probability

def organize_img(img):
  img = img.convert('L') # 'L' = grayscale
  img = img.resize((48,48)) # resizing the img to fit the model (48,48)
  img = np.array(img) #transforming the img into an array in order to normalize it
  img = img / 255 #normaliting the img
  img = img.reshape(1,48,48,1)
  return img

def predict(img, cnn_model):
  prediction = cnn_model.predict(img)
  prediction_output = np.argmax(prediction) #taking the place of the highest propability
  full_prediction = prediction_class_name[prediction_output] # position into emotion name
  return prediction, prediction_output, full_prediction

def create_answer(prediction, prediction_output, full_prediction):
  if prediction[0][prediction_output] < 0.6:
    answer = f"Unknown, I'm only {prediction[0][prediction_output] * 100:.2f}% sure that this emotion is: {full_prediction}"
  else:
    answer = f"I'm {prediction[0][prediction_output] * 100:.2f}% sure that this emotion is: {full_prediction}"
  return answer

face_detector = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def face_locating (img_test):
  face_img = Image.open(img_test)
  face_img = face_img.convert('L')
  PIL_img = face_img # keep the grayscale PIL image before it becomes an array
  face_img = np.array(face_img)
  return face_img,PIL_img

def face_ditector(face_img,PIL_img):
  faces = face_detector.detectMultiScale(face_img)
  if len (faces) == 0:
    print ("I'm sorry, i couldn't ditect a face to analize")
    return PIL_img
  x = faces[0][0]
  y = faces[0][1]
  w = faces[0][2]
  h = faces[0][3]
  croped_img = PIL_img.crop((x, y, x + w, y + h))  # crop to the face box: (left, top, right, bottom); right = x+w, bottom = y+h
  return croped_img

face_img,PIL_img = face_locating(img_test)
croped_img = face_ditector(face_img,PIL_img)
img = organize_img(croped_img)
prediction, prediction_output, full_prediction = predict(img, cnn_model)
answer = create_answer(prediction, prediction_output, full_prediction)
emotions = emotions_probability(prediction_class_name,prediction)
print(emotions)
print(answer)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step
angry: 0.01%
disgust: 0.00%
fear: 1.47%
happy: 0.03%
neutral: 3.71%
sad: 0.18%
surprise: 94.60%

I'm 94.60% sure that this emotion is: surprise


In [27]:
from sklearn.metrics import confusion_matrix, classification_report

eval_set = test_datagen.flow_from_directory(
    test_data, target_size=(48, 48), batch_size=32,
    class_mode='categorical', color_mode='grayscale', shuffle=False)

y_true = eval_set.classes
y_pred = np.argmax(cnn_model.predict(eval_set), axis=1)
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=prediction_class_name))

Found 7042 images belonging to 7 classes.
221/221 ━━━━━━━━━━━━━━━━━━━━ 15s 69ms/step
[[ 351    0    0   56  162   32   37]
 [  14    0    1    7   23   10    2]
 [   5    0   37   19   26    7   73]
 [  37    0    0 1593  124   34   39]
 [  27    0    1  118 2239  152   60]
 [  35    0    9   77  405  317   13]
 [  31    0    7   37  101   13  711]]
              precision    recall  f1-score   support

       angry       0.70      0.55      0.62       638
     disgust       0.00      0.00      0.00        57
        fear       0.67      0.22      0.33       167
       happy       0.84      0.87      0.85      1827
     neutral       0.73      0.86      0.79      2597
         sad       0.56      0.37      0.45       856
    surprise       0.76      0.79      0.77       900

    accuracy                           0.75      7042
   macro avg       0.61      0.52      0.54      7042
weighted avg       0.73      0.75      0.73      7042



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
